# Setup 

Imports and utility functions: 

In [38]:
import sys
import os
import re
import pandas as pd
import json
import urllib.request

sys.path.append(os.getcwd())
from mgrsurvey import loader

def cite(index):
    """
    Take a pandas.Sequence, presumed to be of citation keys, and return a latex citation string
    """
    return f'\cite{{{index.str.cat(sep=", ")}}}'

load spreadsheet data and bibliography, check they align 

In [39]:
dat, pubs = loader.load_mgr_survey("data/new_survey.csv", "data/articles.bib")

# set diffs between the spreadsheet and bibtex indices: should both be empty!
print(dat.index.difference(pubs.index))
print(pubs.index.difference(dat.index))

Index([], dtype='object')
Index([], dtype='object')


Make a `\cite` command for the whole corpus

In [71]:
# cite all the library things, sorted on year then on the citation key
print(
    cite(
        dat.assign(citekey=list(dat.index)) # add a dummy col for the citekeys
        .sort_values(by=[("year", ""), ("citekey", "")]) # now I regret having a multiindex 
        .index
    )
)

\cite{deLeon2004, Flexer2005b, Hu2005, Peeters2005, Hansen2007, Hockman2008, Schuller2008, ForoughmandArabi2009, Itoyama2009, Schnitzer2012, AbbasiLayegh2013, Agarwal2013, Andreas2013, Arenas-Garcia2013, Baniya2013, Baniya2013a, Bernabeu2013, Boonmatham2013, Chathuranga2013a, Chen2013, Conklin2013, Coronel2013, Costa2013, Costa2013a, Dieleman2013, Doudpota2013, Foucard2013, Hamel2013, Haq2013, HaroBerois2013, Hartmann2013, Jao2013, Jensen2013, Koerich2013, Kostek2013a, Kotsifakos2013, Markov2013, Markov2013a, Martel2013, Nakamura2013, Panagakis2013, Piccoli2013, Poria2013, Qin2013, Rodrigues2013, Rosner2013, Scardapane2013, Sturm2013c, Sturm2013d, Sturm2013e, Sturm2013g, Sturm2013h, Vatolkin2013, Vatolkin2013a, Wang2013, Wang2013a, Wu2013, Yang2013, Yeh2013, Zanoni2013, Abesser2014, Ahmad2014, Ajoodha2014, Alexandridis2014, Anden2014, Anglade2014, Antunes2014, Arjannikov2014, Aryafar2014, Banitalebi-Dehkordi2014, Baniya2014, Baniya2014a, Bulos2014, CapoClar2014, Chou2014, Ding2014, Epp

# Publication Types

Total number of publications in corpus:

In [46]:
print("Total pubs", len(dat))

Total pubs 559


Summary of publication types:

In [47]:
dat.function.sum(axis=0).to_frame()

,0
variable,
review,9
evaluation,19
experimental,544


`\cite` commands for each category of publication:

In [45]:
print("Reviews", cite(dat[dat.function.review].index))
print("Evaluations", cite(dat[dat.function.evaluation].index))
print("Experiments", cite(dat[dat.function.experimental].index))

Reviews \cite{Kostek2013a, Sturm2013c, Nasridinov2014, Sturm2014d, Tzanetakis2014, Correa2016, Hoang2018, Narkhede2022, Ozseven2022}
Evaluations \cite{Sturm2013c, Sturm2013e, Sturm2013g, Sturm2013h, Sturm2014c, Sturm2014d, Kereliuk2015, Kereliuk2015a, Schreiber2015, Sturm2015b, Porter2016, Rodriguez-Algarra2016, Sturm2016b, Defferrard2017, Hossain2018, Palmason2018, Rodriguez-Algarra2019, Nie2022, Sturm2015c}
Experiments \cite{deLeon2004, Flexer2005b, Hu2005, Peeters2005, Hansen2007, Hockman2008, Schuller2008, ForoughmandArabi2009, Itoyama2009, Schnitzer2012, AbbasiLayegh2013, Agarwal2013, Andreas2013, Arenas-Garcia2013, Baniya2013, Baniya2013a, Bernabeu2013, Boonmatham2013, Chathuranga2013a, Chen2013, Conklin2013, Coronel2013, Costa2013, Costa2013a, Dieleman2013, Doudpota2013, Foucard2013, Hamel2013, Haq2013, HaroBerois2013, Hartmann2013, Jao2013, Jensen2013, Koerich2013, Kostek2013a, Kotsifakos2013, Markov2013, Markov2013a, Martel2013, Nakamura2013, Panagakis2013, Piccoli2013, Poria2

# Datasets

Get the subset of publications with an experimental component

In [87]:
exp = dat[dat.function.experimental]
print(f'{len(exp)} experimental papers')

544 experimental papers


Summary of top 5 datasets used in experiments: 

In [88]:
exp.datasets.sum(axis=0).sort_values(ascending=False).head(n=5).to_frame()

,0
variable,
gtzan,253
ismir2004,50
fma,36
msd,31
latin,23


## Private datasets

Counts of private dataset usage in experiments:

In [83]:
print(f'{len(exp[exp.private.data])} papers use private data')
only_private = len(exp[(exp.datasets.sum(axis=1) < 1) & (exp.private.data)])
print(f"{only_private} papers use only private data")

176 papers use private data
146 papers use only private data


# Modalities

Summary of modality counts for `experiments`

In [77]:
exp.modality.sum().to_frame()

,0
variable,
music_audio,481
music_symb_data,40
other_lyrics__www__playlists,66
non_western,60


Who uses both audio and symbolic data in `experiments`?

In [90]:
audio_and_sym = exp[exp.modality.music_audio & exp.modality.music_symb_data]
print(f"{len(audio_and_sym)} use both audio and symbolic")
print(cite(audio_and_sym.index))

11 use both audio and symbolic
\cite{Itoyama2009, Piccoli2013, Abesser2014, Anglade2014, Pereira2017, Lee2018b, Pereira2019, Heerde2020, He2022, Zheng2022, Vatolkin2022a}


How many use 'non-Western' music across *the whole corpus*?

In [89]:
all_non_western = dat[dat.modality.non_western]
print(f"Non-western against whole corpus: {len(all_non_western)}")

Non-western against whole corpus: 61


# Experimental Designs 

Summary of doe counts across `experiments`

In [80]:
doe = exp.doe
doe_table = doe.sum(axis=0).sort_values(ascending=False)
doe_table.to_frame()

,0
variable,
classify,513
feature,145
generalize,126
cluster,51
eyeball,35
robust,15
retrieve,8
scale,8
rules,6


Prevalence of `classify`:

In [81]:
print(f"{100 * doe_table.classify / len(doe)}% use Classify")
only_classify = exp[(exp.doe.classify) & (exp.doe.sum(axis=1) == 1)]
print(f"{len(only_classify)} papers use only Classify")

94.30147058823529 % use Classify
264 papers just use Classify


Who's using the less common DoEs?

In [82]:
print("Papers that use Rules:", cite(doe.query("rules").index))
print("Papers that use Compose:", cite(doe.query("compose").index))

Papers that use Rules: \cite{Itoyama2009, Coronel2013, Rompre2017, Korvel2019, Campobello2020, Chang2021}
Papers that use Compose: \cite{Itoyama2009, Sturm2015b, Goienetxea2018, Tokui2020, Tang2022}


# Figures of Merit


Summary of FoMs used in `experiments`

In [92]:
fom = exp.fom.sum(axis=0).sort_values(ascending=False)
fom.to_frame()

,0
variable,
accuracy,448
confusion_matrix,193
conf_discussion,115
precision,105
recall,101
f1,64
roc,38
confusion_instances,9
mean_log_loss,3


How many use a confusion matrix without any discussion? 

In [93]:
print(
    f"\n{fom.confusion_matrix - fom.conf_discussion} papers conf table without discussion"
)


78 papers conf table without discussion


Which authors examine confusion instances?

In [94]:
print(
    "\nThese papers do confusion instances",
    cite(exp[exp.fom.confusion_instances].index),
)


These papers do confusion instances \cite{Sturm2013c, Sturm2013g, Sturm2014, Kereliuk2015, Kereliuk2015a, Sturm2015a, Palmason2018, Campobello2020, Hsu2021}


# Engagement with Critical MIR & SSH Genre Theories

Summary of citations and engagement

In [95]:
dat.curiosity.sum().to_frame()

,0
variable,
cites_critical_mir,163
engages_critical_mir,77
cites_genre_theory,36
engages_genre_theory,10


Who 'engages' with genre theory?

In [ ]:
print("Engage theory:", cite(dat.curiosity.query("engages_genre_theory").index))

How many since 2012 using `gztan` with `classify`? 

In [100]:
gt_class_acc = exp[(exp.year > 2012) & (exp.datasets.gtzan) & (exp.doe.classify)]
print(f"{len(gt_class_acc)} pubs after 2012 using gtzan and classify")

249 pubs after 2012 using gtzan and classify


How many since 2012 using *only* `gtzan`, `classify` and `accuracy`?

In [97]:
# Number of papers after 2012 using gtzan and classify

gt_class_acc_only = dat[
    (dat.year > 2012)
    & (dat.datasets.gtzan)
    & (dat.doe.classify)
    & (dat.fom.accuracy)
    & (dat.datasets.sum(axis=1) == 1)
    & (dat.doe.sum(axis=1) == 1)
    & (dat.fom.sum(axis=1) == 1)
]

print(f"{len(gt_class_acc)} pubs after 2012 using only gtzan, classify and accuracy")


38 pubs after 2012 using gtzan, classify and accuracy


## Critical MIR citations

Load up data about who cited which critical MIR pubs

In [ ]:
# json file generated by PDF scraping script
with open("data/critical_mir_citations.json", "r") as f:
    crit_cites = json.load(f)
cite_records = [(k, vi) for k, v in crit_cites.items() for vi in v]
scraped_crits = pd.DataFrame.from_records(cite_records, columns=["source", "referer"])

# Compare our record of critical citations with what is recorded in the spreadsheet; should be no difference!
# (i.e. everything marked as having a critical mir cite should be represented in scraped_crits)
dat[dat.curiosity.cites_critical_mir].index.symmetric_difference(
    scraped_crits.referer.unique()
)

Who cites these Sturm papers? 

In [27]:
cites_sturm = scraped_crits[
    scraped_crits.source.isin(
        [
            "Sturm2012c",
            "Sturm2012e",
            "Sturm2013g",
            "Sturm2013h",
            "Sturm2014c",
            "Sturm2016b",
        ]
    )
]

pd.DataFrame(
    cites_sturm.groupby(scraped_crits.source)[["referer"]].apply(
        lambda x: (cite(x.referer))
    )
).style

,0
source,
Sturm2012c,"\cite{Bisharad2019, Sturm2015b, Marques2014, Sturm2014d, Weiss2017, Bisharad2019a, Sturm2013c, Koerich2013}"
Sturm2012e,"\cite{Sturm2013g, Nie2022, Sturm2014d, Rosner2018, Sturm2013e, Sturm2014, Schindler2016, Bisharad2019, Vatolkin2022a, Johnson-Roberson2015, Rodriguez-Algarra2016, Sturm2014c, Sturm2013d, Bogdanov2016, Esparza2015, Karunakaran2018, Sturm2015b, Epure2019, Goienetxea2018, OBrien2015, Sturm2013h, Ng2020, Marques2014, Sturm2015a, Weiss2017, Kumar2017, Sturm2013c, Campobello2020, Kereliuk2015, Correa2016, Tavares2018, Costa2017, Nanni2017}"
Sturm2013g,"\cite{Sturm2015b, Sturm2015a, Schindler2016, Prezja2018, Porter2016, Kereliuk2015, Jao2015, McKay2018}"
Sturm2013h,"\cite{Sturm2013g, Sturm2015b, Wibowo2022, Kereliuk2015a, Sturm2015a, Ferraro2018, Ramirez2020, Palmason2018, Kostrzewa2021, Medhat2017a, Castillo2021, deSousa2016, Prezja2018, Kereliuk2015, Lu2016, Prockup2015, Jeong2016, SheikhFathollahi2021, Mujtaba2019, Silva2021a}"
Sturm2016b,"\cite{Choi2017, Rodriguez-Algarra2019, Rodriguez-Algarra2016}"


# Venues

In [461]:
pubs.type.value_counts()

type
conference      349
journal         156
online           20
mathesis         13
incollection     10
phdthesis         9
thesis            1
report            1
Name: count, dtype: int64

In [668]:
# Take conference names and strip out years and 'nth so and so' conference to try and get unique 'venues'
guff_re = re.compile(
    r"\s?(\d?\d(rd|st|nd|th))|\d{4}|(first|second|third|fourth|fifth|sixth|seventh|eighth|ninth|tenth)|\(.+\)|(Proceedings (of (the)?)?)\s?",
    flags=re.IGNORECASE,
)

conferences_rw = pubs[pubs.type == "conference"].venue
conferences_clean = conferences_rw.str.replace(guff_re, repl="", regex=True).str.strip()
conferences_clean_counts = conferences_clean.value_counts().to_frame()
display(HTML(conferences_clean_counts.head(10).to_html()))
print(f"{len(conferences_clean_counts)} distinct conferences")

print(conferences_clean_counts.value_counts())

,count
venue,
International Society for Music Information Retrieval Conference,43
"IEEE International Conference on Acoustics, Speech and Signal Processing",11
International Joint Conference on Neural Networks,7
Interspeech,5
International Workshop on Machine Learning and Music,4
European Signal Processing Conference,4
"IEEE International Conference on Systems, Man, and Cybernetics",4
Anais Do XVIII Simpósio Brasileiro de Computação Musical,4
Asia-Pacific Signal and Information Processing Association Annual Summit and Conference,4


235 distinct conferences
count
1        195
2         26
4          6
3          4
5          1
7          1
11         1
43         1
Name: count, dtype: int64


In [669]:
journal_counts = pubs[pubs.type == "journal"].venue.value_counts()

print(journal_counts[journal_counts >= 4])
journal_counts.value_counts(sort=False)

# print_em_oot = (
#     journal_counts[journal_counts >= 4]
#     .reset_index()
#     .apply(lambda x: print(f"\\textit{{{x[0]}}} ({x[1]})"), axis=1)
# )

print(f"{len(journal_counts)} distinct journals")
print(journal_counts.value_counts())

venue
IEEE Access                                                                  7
International Research Journal of Engineering and Technology                 6
IEEE Signal Processing Letters                                               4
International Journal of Computer Applications                               4
Journal of Intelligent Information Systems                                   4
IEEE Transactions on Multimedia                                              4
Journal of New Music Research                                                4
Transactions of the International Society for Music Information Retrieval    4
Expert Systems with Applications                                             4
Applied Soft Computing                                                       4
Name: count, dtype: int64
\textit{IEEE Access} (7)
\textit{International Research Journal of Engineering and Technology} (6)
\textit{IEEE Signal Processing Letters} (4)
\textit{International Journal of Co

In [465]:
pubs["ismir"] = pubs.venue.str.contains(
    r"ISMIR|international society for music information", case=False
)
pubs["ieee_acm"] = pubs.venue.str.contains(r"IEEE|ACM", case=False)
pubs["icmc_smc_dafx"] = pubs.venue.str.contains(
    r"dafx|international computer music|sound and music computing", case=False
)
pubs[["type", "ismir", "ieee_acm", "icmc_smc_dafx"]].groupby(pubs.type).sum().drop(
    columns=["type"]
)

,ismir,ieee_acm,icmc_smc_dafx
type,,,
conference,45,58,6
incollection,0,0,0
journal,4,23,0
mathesis,0,0,0
online,0,0,0
phdthesis,0,0,0
report,0,0,0
thesis,0,0,0


## Publisher Information 

This block is a bit fiddly and waity-aroundy, and requires to have downloaded crossref's json file of covered journals 

In [818]:
# download the crossref file if needed
crossref_titles_url = "http://ftp.crossref.org/titlelist/titleFile.csv"
urllib.request.urlretrieve(crossref_titles_url, "data/crossref_titleFile_240404.csv")

('data/crossref_titleFile_240404.csv',
 <http.client.HTTPMessage at 0x1571a4490>)

In [433]:
# journal info from crossref
journal_db = pd.read_csv(
    "data/crossref_titleFile_240404.csv",
    skip_blank_lines=True,
    skipinitialspace=True,
    index_col=0,
    header=0,
)

# haha sneaky nan in index
journal_db = journal_db[journal_db.index.notnull()]

The next block takes a few minutes to run whilst we fuzzy match every journal name against the crossref catalogue. There's probably a faster library for this. 

In [490]:
import difflib


def match_journal(x):
    return difflib.get_close_matches(x, journal_db.index, n=1, cutoff=0.6)[0]

journals = pubs[pubs.type == "journal"]
crossref_journals = journals.venue.apply(match_journal)

In [492]:
journals_matched = pd.DataFrame(journals).join(crossref_journals, rsuffix="_matched")[
    ["venue", "venue_matched"]
]
# Get the journal names with the lowest matches. Empirically, 0.95 separates wheat from chaff
journals_matched["ratio"] = journals_matched.apply(
    lambda x: difflib.SequenceMatcher(a=x["venue"], b=x["venue_matched"]).ratio(),
    axis=1,
)
journals_matched[journals_matched.ratio < 0.95].style.background_gradient(
    subset="ratio", cmap="summer_r"
)

,venue,venue_matched,ratio
Chavan2019,International Research Journal of Engineering and Technology,International Journal of Engineering and Technology,0.918919
Chen2016a,Journal of Network Intelligence,Journal of Intelligence,0.851852
Cheng2020a,Engineering Letters,Advanced Engineering Letters,0.808511
Girase2014,Journal of Computing Technologies,Journal of Emerging Technologies,0.861538
Imran2017,"Asian Journal of Engineering, Sciences \& Technology",Journal of Engineering Science and Technology,0.865979
Koparde2021,International Research Journal of Engineering and Technology,International Journal of Engineering and Technology,0.918919
Kothari2022,International Research Journal of Engineering and Technology,International Journal of Engineering and Technology,0.918919
Kumar2017,International Research Journal of Engineering and Technology,International Journal of Engineering and Technology,0.918919
Lee2018a,Journal of Theoretical \& Applied Information Technology,Journal of Geospatial Information Technology,0.800000
Macharla2021,Journal of Innovation Sciences and Sustainable Technologies,Journal of Innovations in Science and Technology,0.803738


In [493]:
print(
    "Journals not indexed by crossref and the count of publications appearing in each"
)
journals_matched[journals_matched.ratio < 0.95].value_counts(subset=["venue"])

Journals not indexed by crossref and the count of publications appearing in each


venue                                                                  
International Research Journal of Engineering and Technology               6
Journal of Theoretical \& Applied Information Technology                   2
Advances in Computer Science Research                                      1
Asian Journal of Engineering, Sciences \& Technology                       1
Engineering Letters                                                        1
International Journal of Scientific Engineering and Technology Research    1
Journal of Computing Technologies                                          1
Journal of Innovation Sciences and Sustainable Technologies                1
Journal of Network Intelligence                                            1
Studia Informatica Pomerania                                               1
Name: count, dtype: int64

In [507]:
# Get top publishers for journals
indexed_journals = journals_matched[
    journals_matched.ratio >= 0.95
].venue_matched.unique()
journal_records = journal_db[journal_db.index.isin(indexed_journals)]
# 'Turkish Online Journal of Qualitative Inquiry' turns out to have multiple entries in the crossref db
journal_records = journal_records[~journal_records.index.duplicated(keep="first")]
print(pd.DataFrame(journal_records.Publisher.value_counts().head(10)).style.to_latex())

\begin{tabular}{lr}
 & count \\
Publisher &  \\
Elsevier  & 11 \\
Springer-Verlag & 9 \\
Institute of Electrical and Electronics Engineers & 8 \\
MDPI AG & 5 \\
Hindawi Limited & 4 \\
Association for Computing Machinery & 3 \\
Institution of Engineering and Technology (IET) & 3 \\
MECS Publisher & 2 \\
Sciedu Press & 2 \\
Informa UK (Taylor & Francis) & 1 \\
\end{tabular}



# Motivations

In [641]:
dat.rationale.sum()

variable
information_overload    150
self_referential        102
dtype: int64

In [642]:
dat.appeals.sum()

variable
industry          106
public_utility     49
dtype: int64